In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

# Path Configuration
from tools.preprocess import *

# Processing context
trait = "Kidney_Papillary_Cell_Carcinoma"
cohort = "GSE68950"

# Input paths
in_trait_dir = "../../input/GEO/Kidney_Papillary_Cell_Carcinoma"
in_cohort_dir = "../../input/GEO/Kidney_Papillary_Cell_Carcinoma/GSE68950"

# Output paths
out_data_file = "../../output/preprocess/Kidney_Papillary_Cell_Carcinoma/GSE68950.csv"
out_gene_data_file = "../../output/preprocess/Kidney_Papillary_Cell_Carcinoma/gene_data/GSE68950.csv"
out_clinical_data_file = "../../output/preprocess/Kidney_Papillary_Cell_Carcinoma/clinical_data/GSE68950.csv"
json_path = "../../output/preprocess/Kidney_Papillary_Cell_Carcinoma/cohort_info.json"


### Step 1: Initial Data Loading

In [ ]:
from tools.preprocess import *
# 1. Identify the paths to the SOFT file and the matrix file
soft_file, matrix_file = geo_get_relevant_filepaths(in_cohort_dir)

# 2. Read the matrix file to obtain background information and sample characteristics data
background_prefixes = ['!Series_title', '!Series_summary', '!Series_overall_design']
clinical_prefixes = ['!Sample_geo_accession', '!Sample_characteristics_ch1']
background_info, clinical_data = get_background_and_clinical_data(matrix_file, background_prefixes, clinical_prefixes)

# 3. Obtain the sample characteristics dictionary from the clinical dataframe
sample_characteristics_dict = get_unique_values_by_row(clinical_data)

# 4. Explicitly print out all the background information and the sample characteristics dictionary
print("Background Information:")
print(background_info)
print("Sample Characteristics Dictionary:")
print(sample_characteristics_dict)


### Step 2: Dataset Analysis and Clinical Feature Extraction

In [ ]:
# 1. Gene Expression Data Availability
# Since the data has gene expression data (based on the Series title mentioning "Affymetrix gene expression project"),
# this dataset likely contains gene expression data.
is_gene_available = True

# 2. Variable Availability and Data Type Conversion

# 2.1 Data Availability
# For trait (kidney cancer), we can use disease state information (row 1)
trait_row = 1

# Looking at sample characteristics, there's no age information
age_row = None

# No gender information available
gender_row = None

# 2.2 Data Type Conversion Functions
def convert_trait(value):
    """
    Convert disease state information to binary trait values.
    1 for Kidney Papillary Cell Carcinoma, 0 for other conditions
    """
    if value is None:
        return None
    
    # Extract the value part after the colon
    if ':' in value:
        value = value.split(':', 1)[1].strip()
    
    # Based on the disease state values, identify kidney papillary cell carcinoma
    # Looking at disease states, we have "papillary carcinoma" and "renal cell carcinoma" 
    # and "papillary transitional cell carcinoma" that could be related to kidney papillary cell carcinoma
    if "papillary" in value.lower() and ("renal" in value.lower() or "kidney" in value.lower() or "transitional cell carcinoma" in value.lower()):
        return 1
    # If just papillary carcinoma and disease location is kidney, that's likely kidney papillary carcinoma
    elif "papillary carcinoma" in value.lower():
        return 1
    # Explicitly mark renal cell carcinoma
    elif "renal cell carcinoma" in value.lower():
        return 1
    elif "papillary transitional cell carcinoma" in value.lower():
        return 1
    # Other conditions
    else:
        return 0

def convert_age(value):
    # No age data available, function provided for consistency
    return None

def convert_gender(value):
    # No gender data available, function provided for consistency
    return None

# 3. Save Metadata
# Trait data is available since trait_row is not None
is_trait_available = trait_row is not None

# Validate and save cohort info
validate_and_save_cohort_info(
    is_final=False,
    cohort=cohort,
    info_path=json_path,
    is_gene_available=is_gene_available,
    is_trait_available=is_trait_available
)

# 4. Clinical Feature Extraction
# Since trait_row is not None, we need to extract clinical features
if trait_row is not None:
    # Extract clinical features
    clinical_features_df = geo_select_clinical_features(
        clinical_df=clinical_data,  # Assuming clinical_data is the input dataframe
        trait=trait,
        trait_row=trait_row,
        convert_trait=convert_trait,
        age_row=age_row,
        convert_age=convert_age,
        gender_row=gender_row,
        convert_gender=convert_gender
    )
    
    # Preview the dataframe
    preview_data = preview_df(clinical_features_df)
    print("Preview of clinical features:")
    print(preview_data)
    
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(out_clinical_data_file), exist_ok=True)
    
    # Save to CSV
    clinical_features_df.to_csv(out_clinical_data_file)
    print(f"Clinical data saved to {out_clinical_data_file}")


### Step 3: Gene Data Extraction

In [ ]:
# 1. Get the file paths for the SOFT file and matrix file
soft_file, matrix_file = geo_get_relevant_filepaths(in_cohort_dir)

# Add diagnostic code to check file content and structure
print("Examining matrix file structure...")
with gzip.open(matrix_file, 'rt') as file:
    table_marker_found = False
    lines_read = 0
    for i, line in enumerate(file):
        lines_read += 1
        if '!series_matrix_table_begin' in line:
            table_marker_found = True
            print(f"Found table marker at line {i}")
            # Read a few lines after the marker to check data structure
            next_lines = [next(file, "").strip() for _ in range(5)]
            print("First few lines after marker:")
            for next_line in next_lines:
                print(next_line)
            break
        if i < 10:  # Print first few lines to see file structure
            print(f"Line {i}: {line.strip()}")
        if i > 100:  # Don't read the entire file
            break
    
    if not table_marker_found:
        print("Table marker '!series_matrix_table_begin' not found in first 100 lines")
    print(f"Total lines examined: {lines_read}")

# 2. Try extracting gene expression data from the matrix file again with better diagnostics
try:
    print("\nAttempting to extract gene data from matrix file...")
    gene_data = get_genetic_data(matrix_file)
    if gene_data.empty:
        print("Extracted gene expression data is empty")
        is_gene_available = False
    else:
        print(f"Successfully extracted gene data with {len(gene_data.index)} rows")
        print("First 20 gene IDs:")
        print(gene_data.index[:20])
        is_gene_available = True
except Exception as e:
    print(f"Error extracting gene data: {str(e)}")
    print("This dataset appears to have an empty or malformed gene expression matrix")
    is_gene_available = False

print(f"\nGene expression data available: {is_gene_available}")

# If data extraction failed, try an alternative approach using pandas directly
if not is_gene_available:
    print("\nTrying alternative approach to read gene expression data...")
    try:
        with gzip.open(matrix_file, 'rt') as file:
            # Skip lines until we find the marker
            for line in file:
                if '!series_matrix_table_begin' in line:
                    break
            
            # Try to read the data directly with pandas
            gene_data = pd.read_csv(file, sep='\t', index_col=0)
            
            if not gene_data.empty:
                print(f"Successfully extracted gene data with alternative method: {gene_data.shape}")
                print("First 20 gene IDs:")
                print(gene_data.index[:20])
                is_gene_available = True
            else:
                print("Alternative extraction method also produced empty data")
    except Exception as e:
        print(f"Alternative extraction failed: {str(e)}")


### Step 4: Gene Identifier Review

In [ ]:
# Examining the gene identifiers in the first few rows
# These are probe IDs from an Affymetrix HT_HG-U133A array, not standard human gene symbols
# IDs like "1007_s_at", "1053_at", "117_at" are Affymetrix probe identifiers
# They need to be mapped to standard human gene symbols

requires_gene_mapping = True


### Step 5: Gene Annotation

In [ ]:
# 1. Extract gene annotation data from the SOFT file
print("Extracting gene annotation data from SOFT file...")
try:
    # Use the library function to extract gene annotation
    gene_annotation = get_gene_annotation(soft_file)
    print(f"Successfully extracted gene annotation data with {len(gene_annotation.index)} rows")
    
    # Preview the annotation DataFrame
    print("\nGene annotation preview (first few rows):")
    print(preview_df(gene_annotation))
    
    # Show column names to help identify which columns we need for mapping
    print("\nColumn names in gene annotation data:")
    print(gene_annotation.columns.tolist())
    
    # Check for relevant mapping columns
    if 'GB_ACC' in gene_annotation.columns:
        print("\nThe dataset contains GenBank accessions (GB_ACC) that could be used for gene mapping.")
        # Count non-null values in GB_ACC column
        non_null_count = gene_annotation['GB_ACC'].count()
        print(f"Number of rows with GenBank accessions: {non_null_count} out of {len(gene_annotation)}")
    
    if 'SPOT_ID' in gene_annotation.columns:
        print("\nThe dataset contains genomic regions (SPOT_ID) that could be used for location-based gene mapping.")
        print("Example SPOT_ID format:", gene_annotation['SPOT_ID'].iloc[0])
    
except Exception as e:
    print(f"Error processing gene annotation data: {e}")
    is_gene_available = False


### Step 6: Gene Identifier Mapping

In [ ]:
# 1. Decide which keys store gene identifiers and gene symbols
# From examining the data:
# - 'ID' column in gene_annotation contains probe identifiers matching those in gene_data (e.g., "1007_s_at")
# - 'Gene Symbol' column contains human gene symbols (e.g., "DDR1", "RFC2")

# 2. Get gene mapping dataframe by extracting the probe ID and gene symbol columns
print("Creating gene mapping dataframe...")
gene_mapping = get_gene_mapping(gene_annotation, prob_col='ID', gene_col='Gene Symbol')
print(f"Created gene mapping with {len(gene_mapping)} rows")
print("\nPreview of gene mapping:")
print(preview_df(gene_mapping))

# 3. Convert probe-level measurements to gene expression data
print("\nApplying gene mapping to convert probe-level data to gene-level data...")
gene_data = apply_gene_mapping(gene_data, gene_mapping)
print(f"Converted gene expression data contains {len(gene_data)} genes")
print("\nPreview of gene expression data (first few genes):")
print(preview_df(gene_data, n=5))

# Save the gene expression data to file
print(f"\nSaving gene expression data to {out_gene_data_file}...")
os.makedirs(os.path.dirname(out_gene_data_file), exist_ok=True)
gene_data.to_csv(out_gene_data_file)
print("Gene expression data saved successfully")


### Step 7: Data Normalization and Linking

In [ ]:
# 1. Normalize gene symbols in the gene expression data
# This was already done and saved in the previous step, verified in the output
# We'll load it to make sure gene_data is available for the rest of the processing
print("\nLoading gene expression data...")
try:
    gene_data = pd.read_csv(out_gene_data_file, index_col=0)
    print(f"Loaded gene data with {gene_data.shape[0]} genes and {gene_data.shape[1]} samples")
except Exception as e:
    print(f"Error loading gene data: {e}")
    gene_data = pd.DataFrame()

# 2. Extract clinical features
print("\nExtracting clinical features...")
try:
    # Get the file paths for the SOFT file and matrix file again
    soft_file, matrix_file = geo_get_relevant_filepaths(in_cohort_dir)
    
    # Obtain background information and clinical dataframe from matrix file
    background_info, clinical_data = get_background_and_clinical_data(
        matrix_file, 
        prefixes_a=['!Series_title', '!Series_summary', '!Series_overall_design'],
        prefixes_b=['!Sample_geo_accession', '!Sample_characteristics_ch1']
    )
    
    # For diagnostic purposes, check the unique values in each row
    sample_characteristics_dict = get_unique_values_by_row(clinical_data)
    print("Sample Characteristics Dictionary:")
    print(sample_characteristics_dict)
    
    # Based on sample_characteristics_dict:
    # Row 1 contains disease state info
    # Row 2 contains disease location
    # We'll use both to identify kidney cancer samples
    trait_row = 2  # Use disease location row
    
    # Define conversion function for the trait (kidney cancer)
    def convert_trait(value):
        """
        Convert disease location to binary trait values based on kidney location.
        1 for kidney samples, 0 for other locations
        """
        if value is None:
            return None
        
        # Extract the value part after the colon
        if ':' in value:
            value = value.split(':', 1)[1].strip()
        
        # Check if it's a kidney sample
        if 'kidney' in value.lower():
            return 1  # Kidney samples as cases
        else:
            return 0  # Other locations as controls

    # Extract clinical features
    selected_clinical_df = geo_select_clinical_features(
        clinical_df=clinical_data,
        trait=trait,
        trait_row=trait_row,
        convert_trait=convert_trait,
        age_row=None,  # No age data
        convert_age=None,
        gender_row=None,  # No gender data
        convert_gender=None
    )
    
    # Save the clinical data
    os.makedirs(os.path.dirname(out_clinical_data_file), exist_ok=True)
    selected_clinical_df.to_csv(out_clinical_data_file)
    print(f"Clinical data extracted and saved to {out_clinical_data_file}")
    
    # For diagnostic purposes
    print("Clinical data preview:")
    print(preview_df(selected_clinical_df))
    
except Exception as e:
    print(f"Error extracting clinical features: {e}")
    selected_clinical_df = pd.DataFrame()

# 3. Link clinical and genetic data
print("\nLinking clinical and genetic data...")
try:
    # Check if both datasets are available
    if not gene_data.empty and not selected_clinical_df.empty:
        # Link clinical and genetic data
        linked_data = geo_link_clinical_genetic_data(selected_clinical_df, gene_data)
        print(f"Linked data shape: {linked_data.shape}")
        
        # 4. Handle missing values
        print("\nHandling missing values...")
        linked_data = handle_missing_values(linked_data, trait)
        print(f"After handling missing values, data shape: {linked_data.shape}")
        
        # 5. Determine whether the trait and demographic features are biased
        print("\nChecking for bias in features...")
        is_biased, linked_data = judge_and_remove_biased_features(linked_data, trait)
    else:
        print("Clinical data or gene data is not available. Cannot proceed with linking.")
        linked_data = pd.DataFrame()
        is_biased = True
except Exception as e:
    print(f"Error in linking data: {e}")
    linked_data = pd.DataFrame()
    is_biased = True

# 6. Final quality validation
print("\nPerforming final validation...")
note = ""
if 'linked_data' not in locals() or linked_data.empty:
    note = "Dataset failed processing: no usable samples remained after filtering."
    is_biased = True
    linked_data = pd.DataFrame()  # Ensure linked_data exists for validation
elif linked_data.shape[0] < 10:
    note = "Dataset has too few samples (<10) after filtering for valid analysis."
    is_biased = True
elif 'is_biased' in locals() and is_biased:
    note = "Dataset has severely biased trait distribution."

is_gene_available = 'gene_data' in locals() and not gene_data.empty
is_trait_available = 'selected_clinical_df' in locals() and not selected_clinical_df.empty

is_usable = validate_and_save_cohort_info(
    is_final=True,
    cohort=cohort,
    info_path=json_path,
    is_gene_available=is_gene_available,
    is_trait_available=is_trait_available,
    is_biased=is_biased if 'is_biased' in locals() else True,
    df=linked_data,
    note=note
)

# 7. Save linked data if usable
if is_usable and 'linked_data' in locals() and not linked_data.empty:
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(out_data_file), exist_ok=True)
    
    # Save linked data
    linked_data.to_csv(out_data_file)
    print(f"Linked data saved to {out_data_file}")
else:
    print(f"Dataset not usable for {trait} association studies. Data not saved.")